# Connect to Elasticsearch

In [2]:
from pprint import pprint
from elasticsearch import Elasticsearch

In [3]:
es=Elasticsearch('http://localhost:9200')
client_info=es.info()
print("Connected to Elasticsearch!!")
pprint(client_info.body)

Connected to Elasticsearch!!
{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'uN3swH3hQG-sv9AINTnPIg',
 'name': 'd274ca1284bc',
 'tagline': 'You Know, for Search',
 'version': {'build_date': '2024-08-05T10:05:34.233336849Z',
             'build_flavor': 'default',
             'build_hash': '1a77947f34deddb41af25e6f0ddb8e830159c179',
             'build_snapshot': False,
             'build_type': 'docker',
             'lucene_version': '9.11.1',
             'minimum_index_compatibility_version': '7.0.0',
             'minimum_wire_compatibility_version': '7.17.0',
             'number': '8.15.0'}}


# Load Dummy Data

In [4]:
import json

with open('data/dummy_data.json', 'r') as f:
    dummy_data = json.load(f)

pprint(dummy_data)

[{'created_on': '2026-01-05',
  'text': 'Elasticsearch is a distributed search and analytics engine built on '
          'top of Apache Lucene. It allows you to store, search, and analyze '
          'large volumes of data quickly.',
  'title': 'Getting Started with Elasticsearch'},
 {'created_on': '2026-02-14',
  'text': 'An inverted index maps content, such as words or numbers, to its '
          'locations in a database file, document, or set of documents, '
          'enabling fast full-text searches.',
  'title': 'Understanding Inverted Indexes'},
 {'created_on': '2026-03-22',
  'text': 'Vector search uses embeddings to represent data as points in a '
          'high-dimensional space, allowing similarity search based on '
          'semantic meaning rather than exact keyword matches.',
  'title': 'Introduction to Vector Search'},
 {'created_on': '2026-05-01',
  'text': 'FastAPI is a modern, fast web framework for building APIs with '
          'Python based on standard type hints

## Create Index

In [5]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})

## Specify the number of shards and replicas

In [6]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index',
            settings={
                "index":{
                "number_of_shards": 3,
                "number_of_replicas": 2
            }
        }

)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})

## Insert into documents

In [7]:
document = {
    "title": "Elasticsearch Basics",
    "text": "Elasticsearch is a distributed, RESTful search and analytics engine capable of solving a growing number of use cases.",
    "created_on":"2026-03-24",
}

In [8]:
response = es.index(index='my_index', document=document)
response

ObjectApiResponse({'_index': 'my_index', '_id': 'BOtiwKABfP5tZT35KL5V', '_version': 1, 'result': 'created', '_shards': {'total': 3, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

In [9]:
print(response['result'])

created


In [10]:
print(response['_shards'])

{'total': 3, 'successful': 1, 'failed': 0}


In [11]:
print(response['_id'])

BOtiwKABfP5tZT35KL5V


In [12]:
print(response['_index'])

my_index


## Creating multiple documents

In [13]:
def insert_document(document):
    response = es.index(index='my_index', body=document)
    return response


def print_info(response):
    print(f"""Document ID: {response['_id']} is '{
          response["result"]}' and is split into {response['_shards']['total']} shards.""")


In [14]:
for document in dummy_data:
    response = insert_document(document)
    print_info(response)

Document ID: BetiwKABfP5tZT35NL58 is 'created' and is split into 3 shards.
Document ID: ButiwKABfP5tZT35NL6I is 'created' and is split into 3 shards.
Document ID: B-tiwKABfP5tZT35NL6P is 'created' and is split into 3 shards.
Document ID: COtiwKABfP5tZT35NL6V is 'created' and is split into 3 shards.
Document ID: CetiwKABfP5tZT35NL6a is 'created' and is split into 3 shards.
Document ID: CutiwKABfP5tZT35NL6e is 'created' and is split into 3 shards.


## Print mapping

Identifies the types of each field

In [15]:
index_mapping = es.indices.get_mapping(index='my_index')
pprint(index_mapping["my_index"]["mappings"]["properties"])

{'created_on': {'type': 'date'},
 'text': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
          'type': 'text'},
 'title': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
           'type': 'text'}}


## Manual mapping

Manual mapping type to each field

In [16]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

mapping = {
    'properties': {
        'created_on': {'type': 'date'},
        'text': {
            'type': 'text',
            'fields': {
                'keyword': {
                    'type': 'keyword',
                    'ignore_above': 256
                }
            }
        },
        'title': {
            'type': 'text',
            'fields': {
                'keyword': {
                    'type': 'keyword',
                    'ignore_above': 256
                }
            }
        }
    }
}

es.indices.put_mapping(index='my_index', body=mapping)

index_mapping = es.indices.get_mapping(index='my_index')
pprint(index_mapping["my_index"]["mappings"]["properties"])

{'created_on': {'type': 'date'},
 'text': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
          'type': 'text'},
 'title': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
           'type': 'text'}}


# Common types of data

## Binary

In [17]:
es.indices.delete(index='binary_index', ignore_unavailable=True)
es.indices.create(
    index='binary_index',
    mappings={
        "properties": {
            "image_data": {
                "type": "binary"
            }
        }
    }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'binary_index'})

In [18]:
import base64

image_path = "images/image.png"
with open(image_path, "rb") as image_file:
    image_bytes = image_file.read()
    image_base64 = base64.b64encode(image_bytes).decode("utf-8")

image_base64[:100]

'iVBORw0KGgoAAAANSUhEUgAABjUAAAPUCAYAAADlo7uJAAAAAXNSR0IArs4c6QAAAFZlWElmTU0AKgAAAAgAAYdpAAQAAAABAAAA'

In [19]:
document = {
    "image_data": image_base64
}
response = es.index(index='binary_index', body=document)
response

ObjectApiResponse({'_index': 'binary_index', '_id': 'C-tiwKABfP5tZT35PL4l', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

In [20]:
pprint(es.indices.get_mapping(index='binary_index'))

ObjectApiResponse({'binary_index': {'mappings': {'properties': {'image_data': {'type': 'binary'}}}}})


## Others

In [21]:
es.indices.delete(index='other_common_data_types_index',
                  ignore_unavailable=True)
es.indices.create(
    index='other_common_data_types_index',
    mappings={
        "properties": {
            "book_reference": {
                "type": "keyword"
            },
            "price": {
                "type": "float"
            },
            "publish_date": {
                "type": "date"
            },
            "is_available": {
                "type": "boolean"
            },
        }
    }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'other_common_data_types_index'})

In [22]:
document = {
    "book_reference": "978-1617294433",
    "price": 44.99,
    "publish_date": "2021-06-30",
    "is_available": True
}
response = es.index(index='other_common_data_types_index', body=document)
response

ObjectApiResponse({'_index': 'other_common_data_types_index', '_id': 'DOtiwKABfP5tZT35QL4w', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

## Object types

### Object

In [23]:
es.indices.delete(index='object_index', ignore_unavailable=True)
es.indices.create(
    index='object_index',
    mappings={
        "properties": {
            "author": {
                "properties": {
                    "first_name": {
                        "type": "text"
                    },
                    "last_name": {
                        "type": "text"
                    }
                }
            }
        }
    }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'object_index'})

In [24]:
document = {
    "author": {
        "first_name": "George",
        "last_name": "Kuncheria"
    }
}
response = es.index(index='object_index', body=document)
response

ObjectApiResponse({'_index': 'object_index', '_id': 'DetiwKABfP5tZT35Q77k', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

###  Flattened object

In [25]:
es.indices.delete(index='flattened_object_index', ignore_unavailable=True)
es.indices.create(
    index='flattened_object_index',
    mappings={
        "properties": {
            "author": {
                "type": "flattened"
            }
        }
    }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'flattened_object_index'})

In [26]:
document = {
    "author": {
        "first_name": "George",
        "last_name": "Kuncheria"
    }
}
response = es.index(index='flattened_object_index', body=document)
response

ObjectApiResponse({'_index': 'flattened_object_index', '_id': 'DutiwKABfP5tZT35Rr4u', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

### Nested object

In [27]:
es.indices.delete(index='nested_object_index', ignore_unavailable=True)
es.indices.create(
    index='nested_object_index',
    mappings={
        "properties": {
            "user": {
                "type": "nested",
            }
        }
    }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'nested_object_index'})

In [28]:
documents = [
    {
        "first": "John",
        "last": "Smith"
    },
    {
        "first": "Imad",
        "last": "Saddik"
    }
]
response = es.index(index='nested_object_index', body={"user": documents})
response

ObjectApiResponse({'_index': 'nested_object_index', '_id': 'D-tiwKABfP5tZT35SL4p', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

##  Text search types

### Text

In [29]:
es.indices.delete(index='text_index', ignore_unavailable=True)
es.indices.create(
    index='text_index',
    mappings={
        "properties": {
            "email_body": {
                "type": "text"
            }
        }
    }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'text_index'})

In [30]:
document = {
    "email_body": "Hello, this is a test email."
}
response = es.index(index='text_index', body=document)
response

ObjectApiResponse({'_index': 'text_index', '_id': 'EOtiwKABfP5tZT35S74y', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

### Completion

In [31]:
es.indices.delete(index='text_completion_index', ignore_unavailable=True)
es.indices.create(
    index='text_completion_index',
    mappings={
        "properties": {
            "suggest": {
                "type": "completion"
            }
        }
    }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'text_completion_index'})

In [32]:
document_1 = {
    "suggest": {
        "input": ["Mars", "Planet"]
    }
}

document_2 = {
    "suggest": {
        "input": ["Andromeda", "Galaxy"]
    }
}

es.index(index='text_completion_index', body=document_1)
es.index(index='text_completion_index', body=document_2)

ObjectApiResponse({'_index': 'text_completion_index', '_id': 'EutiwKABfP5tZT35Tr4t', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 1, '_primary_term': 1})

## Spatial data types

### Geo point

In [33]:
es.indices.delete(index='geo_point_index', ignore_unavailable=True)
es.indices.create(
    index='geo_point_index',
    mappings={
        "properties": {
            "location": {
                "type": "geo_point"
            }
        }
    }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'geo_point_index'})

In [34]:
document = {
    "text": "Geopoint as an object using GeoJSON format",
    "location": {
        "type": "Point",
        "coordinates": [
            -71.34,
            41.12
        ]
    }
}
response = es.index(index='geo_point_index', body=document)
response

ObjectApiResponse({'_index': 'geo_point_index', '_id': 'E-tiwKABfP5tZT35Ub7h', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

### Geo shape

In [35]:
es.indices.delete(index='geo_shape_index', ignore_unavailable=True)
es.indices.create(
    index='geo_shape_index',
    mappings={
        "properties": {
            "location": {
                "type": "geo_shape"
            }
        }
    }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'geo_shape_index'})

In [36]:
document_1 = {
    "location": {
        "type": "LineString",
        "coordinates": [
            [
                -77.03653,
                38.897676
            ],
            [
                -77.009051,
                38.889939
            ]
        ]
    }
}
document_2 = {
    "location": {
        "type": "Polygon",
        "coordinates": [
            [
                [100, 0],
                [101, 0],
                [101, 1],
                [100, 1],
                [100, 0],
            ],
            [
                [100.2, 0.2],
                [100.8, 0.2],
                [100.8, 0.8],
                [100.2, 0.8],
                [100.2, 0.2],
            ]
        ]
    }
}

es.index(index='geo_shape_index', body=document_1)
es.index(index='geo_shape_index', body=document_2)

ObjectApiResponse({'_index': 'geo_shape_index', '_id': 'FetiwKABfP5tZT35Vb4T', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 1, '_primary_term': 1})

### Point

In [37]:
es.indices.delete(index='point_index', ignore_unavailable=True)
es.indices.create(
    index='point_index',
    mappings={
        "properties": {
            "location": {
                "type": "point"
            }
        }
    }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'point_index'})

In [38]:
document = {
    "location": {
        "type": "Point",
        "coordinates": [
            -71.34,
            41.12
        ]
    }
}

response = es.index(index='point_index', body=document)
response

ObjectApiResponse({'_index': 'point_index', '_id': 'FutiwKABfP5tZT35Wr4o', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

## Delete Operation

In [39]:
from tqdm import tqdm

document_ids = []
dummy_data = json.load(open("data/dummy_data.json"))
for document in tqdm(dummy_data, total=len(dummy_data)):
    response = es.index(index='my_index', body=document)
    document_ids.append(response['_id'])

100%|██████████| 6/6 [00:00<00:00, 76.81it/s]


In [40]:
document_ids

['F-tiwKABfP5tZT35Xb7s',
 'GOtiwKABfP5tZT35Xr4Y',
 'GetiwKABfP5tZT35Xr4d',
 'GutiwKABfP5tZT35Xr4t',
 'G-tiwKABfP5tZT35Xr4x',
 'HOtiwKABfP5tZT35Xr4z']

In [41]:
response = es.delete(index='my_index', id=document_ids[0])

In [42]:
from pprint import pprint

pprint(response.body)

{'_id': 'F-tiwKABfP5tZT35Xb7s',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 6,
 '_shards': {'failed': 0, 'successful': 1, 'total': 2},
 '_version': 2,
 'result': 'deleted'}


This example below shows that the delete operation fails when providing it with an ID that does not exist in the index.

In [43]:
try:
    response = es.delete(index='my_index', id="id")
except Exception as e:
    print(e)

NotFoundError(404, "{'_index': 'my_index', '_id': 'id', '_version': 1, 'result': 'not_found', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 7, '_primary_term': 1}")


## Get documents

In [44]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})

In [46]:
document_ids = []
dummy_data = json.load(open("data/dummy_data.json"))
for document in tqdm(dummy_data, total=len(dummy_data)):
    response = es.index(index='my_index', body=document)
    document_ids.append(response['_id'])

100%|██████████| 6/6 [00:00<00:00, 113.88it/s]


In [48]:
document_ids

['HetiwKABfP5tZT358b58',
 'HutiwKABfP5tZT358b6e',
 'H-tiwKABfP5tZT358b6h',
 'IOtiwKABfP5tZT358b6k',
 'IetiwKABfP5tZT358b6n',
 'IutiwKABfP5tZT358b6q']

In [49]:
response = es.get(index='my_index', id=document_ids[0])

In [50]:
pprint(response.body)

{'_id': 'HetiwKABfP5tZT358b58',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 0,
 '_source': {'created_on': '2026-01-05',
             'text': 'Elasticsearch is a distributed search and analytics '
                     'engine built on top of Apache Lucene. It allows you to '
                     'store, search, and analyze large volumes of data '
                     'quickly.',
             'title': 'Getting Started with Elasticsearch'},
 '_version': 1,
 'found': True}


In [51]:
try:
    response = es.get(index='my_index', id="id")
except Exception as e:
    print(e)

NotFoundError(404, "{'_index': 'my_index', '_id': 'id', 'found': False}")


## Count documents

In [53]:
response = es.count(index='my_index')
count = response["count"]

print(f"The number of documents in the index is {count}")

The number of documents in the index is 6


This example below shows how to use the query parameter to match certain criteria.

In [55]:
query = {
    "range": {
        "created_on": {
            "gte": "2026-01-01",
            "lte": "2026-05-30",
            "format": "yyyy-MM-dd"
        }
    }
}

response = es.count(index='my_index', query=query)
count = response["count"]

print(f"The number of documents in the index is {count}")

The number of documents in the index is 4


## Exists API

In [56]:
response = es.indices.exists(index='my_index')
response.body

True

Check if document id exist in index

In [57]:
response = es.exists(index='my_index', id=document_ids[0])
response.body

True

## Index Documents

In [58]:

es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})

In [60]:
document_ids = []
dummy_data = json.load(open("data/dummy_data.json"))
for document in tqdm(dummy_data, total=len(dummy_data)):
    response = es.index(index='my_index', body=document)
    document_ids.append(response['_id'])

100%|██████████| 6/6 [00:00<00:00, 108.97it/s]


In [61]:
document_ids

['I-s3waABfP5tZT35Z76d',
 'JOs3waABfP5tZT35Z766',
 'Jes3waABfP5tZT35Z76-',
 'Jus3waABfP5tZT35Z77B',
 'J-s3waABfP5tZT35Z77D',
 'KOs3waABfP5tZT35Z77F']

### If documents exists in the index

#### Update an existing field

In [63]:
# Document Id[0]

response = es.get(index='my_index', id=document_ids[0])
response.body['_source']['title']

'Getting Started with Elasticsearch'

In [64]:
response = es.update(
    index="my_index",
    id=document_ids[0],
    script={
        "source": "ctx._source.title = params.title",
        "params": {
            "title": "Getting Started with Redis"
        }
    },
)
pprint(response.body)

{'_id': 'I-s3waABfP5tZT35Z76d',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 6,
 '_shards': {'failed': 0, 'successful': 1, 'total': 2},
 '_version': 2,
 'result': 'updated'}


In [67]:
# Updated title
response= es.get(index='my_index', id=document_ids[0])
response.body['_source']['title']

'Getting Started with Redis'

#### Add a new field

##### Method 1

In [68]:
response = es.update(
    index="my_index",
    id=document_ids[0],
    script={
        "source": "ctx._source.new_field = 'dummy_value'",
    },
)
pprint(response.body)

{'_id': 'I-s3waABfP5tZT35Z76d',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 7,
 '_shards': {'failed': 0, 'successful': 1, 'total': 2},
 '_version': 3,
 'result': 'updated'}


In [69]:
response = es.get(index='my_index', id=document_ids[0])
pprint(response.body)

{'_id': 'I-s3waABfP5tZT35Z76d',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 7,
 '_source': {'created_on': '2026-01-05',
             'new_field': 'dummy_value',
             'text': 'Elasticsearch is a distributed search and analytics '
                     'engine built on top of Apache Lucene. It allows you to '
                     'store, search, and analyze large volumes of data '
                     'quickly.',
             'title': 'Getting Started with Redis'},
 '_version': 3,
 'found': True}


##### Method 2 (doc)

In [70]:
response = es.update(
    index="my_index",
    id=document_ids[0],
    doc={
        "new_value_2": "dummy_value_2",
    },
)
pprint(response.body)

{'_id': 'I-s3waABfP5tZT35Z76d',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 8,
 '_shards': {'failed': 0, 'successful': 1, 'total': 2},
 '_version': 4,
 'result': 'updated'}


In [71]:
response = es.get(index='my_index', id=document_ids[0])
pprint(response.body)

{'_id': 'I-s3waABfP5tZT35Z76d',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 8,
 '_source': {'created_on': '2026-01-05',
             'new_field': 'dummy_value',
             'new_value_2': 'dummy_value_2',
             'text': 'Elasticsearch is a distributed search and analytics '
                     'engine built on top of Apache Lucene. It allows you to '
                     'store, search, and analyze large volumes of data '
                     'quickly.',
             'title': 'Getting Started with Redis'},
 '_version': 4,
 'found': True}


#### Remove a field

In [72]:
response = es.update(
    index="my_index",
    id=document_ids[0],
    script={
        "source": "ctx._source.remove('new_field')",
    },
)
pprint(response.body)

{'_id': 'I-s3waABfP5tZT35Z76d',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 9,
 '_shards': {'failed': 0, 'successful': 1, 'total': 2},
 '_version': 5,
 'result': 'updated'}


In [73]:
response = es.get(index='my_index', id=document_ids[0])
pprint(response.body)

{'_id': 'I-s3waABfP5tZT35Z76d',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 9,
 '_source': {'created_on': '2026-01-05',
             'new_value_2': 'dummy_value_2',
             'text': 'Elasticsearch is a distributed search and analytics '
                     'engine built on top of Apache Lucene. It allows you to '
                     'store, search, and analyze large volumes of data '
                     'quickly.',
             'title': 'Getting Started with Redis'},
 '_version': 5,
 'found': True}


###  If documents doesn't exist in the index

We use `doc_as_upsert` to tell Elasticsearch that if the document does not exist, it should be inserted as a new document.

In [74]:
response = es.update(
    index="my_index",
    id="1",
    doc={
        "book_id": 1234,
        "book_name": "A book",
    },
    doc_as_upsert=True,
)

In [75]:
pprint(response.body)

{'_id': '1',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 10,
 '_shards': {'failed': 0, 'successful': 1, 'total': 2},
 '_version': 1,
 'result': 'created'}


In [80]:
response=es.count(index='my_index')
print(response["count"])

7


### Without BULK API

In [81]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})

In [82]:
import json
from tqdm import tqdm


document_ids = []
dummy_data = json.load(open("data/dummy_data.json"))
for document in tqdm(dummy_data, total=len(dummy_data)):
    response = es.index(index='my_index', body=document)
    document_ids.append(response['_id'])

100%|██████████| 6/6 [00:00<00:00, 58.23it/s]


Let's update the first and second documents

In [83]:
from pprint import pprint

response = es.update(
    index="my_index",
    id=document_ids[0],
    script={
        "source": "ctx._source.title = params.title",
        "params": {
            "title": "New Title"
        }
    },
)
pprint(response.body)

{'_id': 'KetpwaABfP5tZT35Kb6J',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 6,
 '_shards': {'failed': 0, 'successful': 1, 'total': 2},
 '_version': 2,
 'result': 'updated'}


In [84]:
response = es.update(
    index="my_index",
    id=document_ids[1],
    script={
        "source": "ctx._source.new_field = 'dummy_value'",
    },
)
pprint(response.body)

{'_id': 'KutpwaABfP5tZT35Kb7e',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 7,
 '_shards': {'failed': 0, 'successful': 1, 'total': 2},
 '_version': 2,
 'result': 'updated'}


Let's delete the third document

In [85]:

response = es.delete(index="my_index", id=document_ids[2])
pprint(response.body)

{'_id': 'K-tpwaABfP5tZT35Kb7h',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 8,
 '_shards': {'failed': 0, 'successful': 1, 'total': 2},
 '_version': 2,
 'result': 'deleted'}


We executed each operation one at a time, with each action requiring a separate API call. This approach is slow and inefficient. Now, let’s see how to accomplish the same task using the bulk API.

### BULK API

In [86]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})

In [87]:
response = es.bulk(
    operations=[
        # Action 1
        {
            "index": {
                "_index": "my_index",
                "_id": "1"
            }
        },
        # Source 1
        {
            "title": "Sample Title 1",
            "text": "This is the first sample document text.",
            "created_on": "2024-09-22"
        },
        # Action 2
        {
            "index": {
                "_index": "my_index",
                "_id": "2"
            }
        },
        # Source 2
        {
            "title": "Sample Title 2",
            "text": "Here is another example of a document.",
            "created_on": "2024-09-24"
        },
        # Action 3
        {
            "index": {
                "_index": "my_index",
                "_id": "3"
            }
        },
        # Source 3
        {
            "title": "Sample Title 3",
            "text": "The content of the third document goes here.",
            "created_on": "2024-09-24"
        },
        # Action 4
        {
            "update": {
                "_id": "1",
                "_index": "my_index"
            }
        },
        # Source 4
        {
            "doc": {
                "title": "New Title"
            }
        },
        # Action 5
        {
            "update": {
                "_id": "2",
                "_index": "my_index"
            }
        },
        # Source 5
        {
            "doc": {
                "new_field": "dummy_value"
            }
        },
        # Action 6
        {
            "delete": {
                "_index": "my_index",
                "_id": "3"
            }
        },
    ],
)

pprint(response.body)

{'errors': False,
 'items': [{'index': {'_id': '1',
                      '_index': 'my_index',
                      '_primary_term': 1,
                      '_seq_no': 0,
                      '_shards': {'failed': 0, 'successful': 1, 'total': 2},
                      '_version': 1,
                      'result': 'created',
                      'status': 201}},
           {'index': {'_id': '2',
                      '_index': 'my_index',
                      '_primary_term': 1,
                      '_seq_no': 1,
                      '_shards': {'failed': 0, 'successful': 1, 'total': 2},
                      '_version': 1,
                      'result': 'created',
                      'status': 201}},
           {'index': {'_id': '3',
                      '_index': 'my_index',
                      '_primary_term': 1,
                      '_seq_no': 2,
                      '_shards': {'failed': 0, 'successful': 1, 'total': 2},
                      '_version': 1,
        

If `errors` is `False`, it means the bulk API successfully executed all the actions.

In [88]:
response.body["errors"]

False